# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [1]:
import pandas as pd

data = pd.read_csv("auto_dataset.csv")
data.head()


,brand,model,vehicleType,gearbox,fuelType,notRepairedDamage,powerPS,kilometer,autoAgeMonths,price
0,volkswagen,golf,kleinwagen,manuell,benzin,nein,75,150000,177,1500
1,skoda,fabia,kleinwagen,manuell,diesel,nein,69,90000,93,3600
2,bmw,3er,limousine,manuell,benzin,ja,102,150000,246,650
3,peugeot,2_reihe,cabrio,manuell,benzin,nein,109,150000,140,2200
4,mazda,3_reihe,limousine,manuell,benzin,nein,105,150000,136,2000


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [2]:
X = pd.get_dummies(data.drop(columns="price"), dtype=float)
y = data["price"].to_numpy(dtype=float)

print("Признаков после кодирования:", X.shape[1])


Признаков после кодирования: 210


3. Разбейте датасет на train val test в отношении 8:1:1

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train_df, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_val_df, X_test_df, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df)
X_val = scaler.transform(X_val_df)
X_test = scaler.transform(X_test_df)

X_train = np.column_stack((np.ones(len(X_train)), X_train))
X_val = np.column_stack((np.ones(len(X_val)), X_val))
X_test = np.column_stack((np.ones(len(X_test)), X_test))

y_mean = y_train.mean()
y_std = y_train.std()
y_train_scaled = (y_train - y_mean) / y_std

print(X_train.shape, X_val.shape, X_test.shape)


(800, 211) (100, 211) (100, 211)


4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

learning_rates = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]

n_train = len(X_train)
G = X_train.T @ X_train / n_train
c = X_train.T @ y_train_scaled / n_train


def predict_price(X, w):
    return (X @ w) * y_std + y_mean


def mse(X, y, w):
    pred = predict_price(X, w)
    return float(np.mean((pred - y) ** 2))


def r2(X, y, w):
    return float(r2_score(y, predict_price(X, w)))


def fit_optimizer(method, lr, decay=False, seed=42):
    rng = np.random.default_rng(seed)
    n, d = X_train.shape
    max_iter = 12000 if method == "sag" else 1500
    w = np.zeros(d)
    history = []
    steps = []

    momentum = np.zeros(d)
    m = np.zeros(d)
    v = np.zeros(d)

    if method == "sag":
        saved_gradients = np.zeros((n, d))
        mean_gradient = np.zeros(d)

    for k in range(max_iter):
        eta = lr / np.sqrt(1 + k) if decay else lr

        if method == "vgd":
            grad = 2 * (G @ w - c)
            w -= eta * grad

        elif method == "sgd":
            idx = rng.choice(n, size=min(32, n), replace=False)
            xb = X_train[idx]
            yb = y_train_scaled[idx]
            grad = 2 / len(idx) * xb.T @ (xb @ w - yb)
            w -= eta * grad

        elif method == "sag":
            j = rng.integers(n)
            old_grad = saved_gradients[j].copy()
            new_grad = 2 * (X_train[j] @ w - y_train_scaled[j]) * X_train[j]
            saved_gradients[j] = new_grad
            mean_gradient += (new_grad - old_grad) / n
            w -= eta * mean_gradient

        elif method == "momentum":
            grad = 2 * (G @ w - c)
            momentum = 0.9 * momentum + eta * grad
            w -= momentum

        elif method == "adam":
            grad = 2 * (G @ w - c)
            m = 0.9 * m + 0.1 * grad
            v = 0.999 * v + 0.001 * grad ** 2
            m_hat = m / (1 - 0.9 ** (k + 1))
            v_hat = v / (1 - 0.999 ** (k + 1))
            w -= eta * m_hat / (np.sqrt(v_hat) + 1e-8)

        if not np.all(np.isfinite(w)) or np.linalg.norm(w) > 1e8:
            break

        if k % 100 == 0 or k == max_iter - 1:
            history.append(mse(X_val, y_val, w))
            steps.append(k + 1)

    return w, history, steps, k + 1


def search_step(method, decay=False):
    rows = []
    variants = []

    for lr in learning_rates:
        w, history, steps, iterations = fit_optimizer(method, lr, decay)
        row = {
            "lambda": lr,
            "Loss_train": mse(X_train, y_train, w),
            "Loss_val": mse(X_val, y_val, w),
            "R2_train": r2(X_train, y_train, w),
        }
        rows.append(row)
        variants.append((w, history, steps, iterations))

    table = pd.DataFrame(rows)
    best_index = table["Loss_val"].replace([np.inf, -np.inf], np.nan).idxmin()
    best_row = table.loc[best_index]
    w, history, steps, iterations = variants[best_index]

    return {
        "method": method,
        "decay": decay,
        "lambda": float(best_row["lambda"]),
        "loss_train": float(best_row["Loss_train"]),
        "loss_val": float(best_row["Loss_val"]),
        "r2_train": float(best_row["R2_train"]),
        "loss_test": mse(X_test, y_test, w),
        "r2_test": r2(X_test, y_test, w),
        "iterations": iterations,
        "history": history,
        "steps": steps,
        "table": table,
    }


def show_search(result):
    display(result["table"])
    if result["decay"]:
        print("Лучший шаг:", f'{result["lambda"]} / sqrt(1 + k)')
    else:
        print("Лучший шаг:", result["lambda"])
    print("Loss val:", round(result["loss_val"], 2))

vgd_const = search_step("vgd", decay=False)
show_search(vgd_const)


,lambda,Loss_train,Loss_val,R2_train
0,0.00001,5.441562e+07,4.940461e+07,1.059242e-01
1,0.00010,2.695696e+07,2.517332e+07,5.570837e-01
2,0.00100,1.336897e+07,1.557103e+07,7.803411e-01
3,0.01000,1.238784e+07,1.751604e+07,7.964616e-01
4,0.10000,1.203073e+07,1.797435e+07,8.023291e-01
5,1.00000,7.672184e+25,7.488405e+25,-1.260578e+18


Лучший шаг: 0.001
Loss val: 15571032.17


5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [5]:
vgd_decay = search_step("vgd", decay=True)
show_search(vgd_decay)


,lambda,Loss_train,Loss_val,R2_train
0,0.00001,6.050706e+07,5.503387e+07,0.005839
1,0.00010,5.745106e+07,5.220220e+07,0.056050
2,0.00100,3.719194e+07,3.390894e+07,0.388918
3,0.01000,1.438496e+07,1.543016e+07,0.763648
4,0.10000,1.257904e+07,1.727300e+07,0.793320
5,1.00000,1.205989e+07,1.770147e+07,0.801850


Лучший шаг: 0.01 / sqrt(1 + k)
Loss val: 15430155.65


6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [6]:
sgd_const = search_step("sgd", decay=False)
show_search(sgd_const)


,lambda,Loss_train,Loss_val,R2_train
0,0.00001,5.434200e+07,4.943276e+07,1.071338e-01
1,0.00010,2.695074e+07,2.529933e+07,5.571858e-01
2,0.00100,1.345308e+07,1.555402e+07,7.789591e-01
3,0.01000,1.293548e+07,1.778459e+07,7.874635e-01
4,0.10000,6.810221e+25,1.835889e+24,-1.118953e+18
5,1.00000,2.028271e+24,3.204632e+24,-3.332550e+16


Лучший шаг: 0.001
Loss val: 15554023.03


7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [7]:
sgd_decay = search_step("sgd", decay=True)
show_search(sgd_decay)


,lambda,Loss_train,Loss_val,R2_train
0,0.00001,6.049396e+07,5.502994e+07,6.053984e-03
1,0.00010,5.733252e+07,5.216849e+07,5.799803e-02
2,0.00100,3.676819e+07,3.388567e+07,3.958803e-01
3,0.01000,1.449348e+07,1.553374e+07,7.618650e-01
4,0.10000,1.276284e+07,1.730803e+07,7.903001e-01
5,1.00000,9.192513e+24,7.818460e+23,-1.510376e+17


Лучший шаг: 0.01 / sqrt(1 + k)
Loss val: 15533740.93


8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [8]:
sag_const = search_step("sag", decay=False)
show_search(sag_const)


,lambda,Loss_train,Loss_val,R2_train
0,0.00001,3.083115e+07,2.838912e+07,4.934288e-01
1,0.00010,1.358157e+07,1.530870e+07,7.768480e-01
2,0.00100,3.348967e+15,2.177217e+14,-5.502520e+07
3,0.01000,7.813016e+23,2.273293e+22,-1.283718e+16
4,0.10000,1.217671e+24,1.067894e+22,-2.000695e+16
5,1.00000,9.511873e+23,2.408578e+23,-1.562848e+16


Лучший шаг: 0.0001
Loss val: 15308700.42


9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [9]:
sag_decay = search_step("sag", decay=True)
show_search(sag_decay)


,lambda,Loss_train,Loss_val,R2_train
0,0.00001,6.008663e+07,5.464029e+07,1.274662e-02
1,0.00010,5.372996e+07,4.874736e+07,1.171898e-01
2,0.00100,2.495849e+07,2.352266e+07,5.899195e-01
3,0.01000,1.317644e+07,1.592713e+07,7.835044e-01
4,0.10000,1.154374e+19,8.251752e+17,-1.896693e+11
5,1.00000,7.728039e+23,1.352782e+23,-1.269755e+16


Лучший шаг: 0.01 / sqrt(1 + k)
Loss val: 15927134.71


10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [10]:
momentum_const = search_step("momentum", decay=False)
show_search(momentum_const)


,lambda,Loss_train,Loss_val,R2_train
0,0.00001,2.698295e+07,2.519905e+07,5.566567e-01
1,0.00010,1.336842e+07,1.557228e+07,7.803502e-01
2,0.00100,1.238860e+07,1.751927e+07,7.964492e-01
3,0.01000,1.203061e+07,1.797690e+07,8.023310e-01
4,0.10000,1.202827e+07,1.820486e+07,8.023694e-01
5,1.00000,3.229439e+24,3.147557e+24,-5.306130e+16


Лучший шаг: 0.0001
Loss val: 15572281.36


11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [11]:
momentum_decay = search_step("momentum", decay=True)
show_search(momentum_decay)


,lambda,Loss_train,Loss_val,R2_train
0,0.00001,5.745962e+07,5.221012e+07,0.055910
1,0.00010,3.717821e+07,3.389873e+07,0.389143
2,0.00100,1.436261e+07,1.540748e+07,0.764015
3,0.01000,1.257814e+07,1.728700e+07,0.793335
4,0.10000,1.205892e+07,1.770357e+07,0.801866
5,1.00000,1.202827e+07,1.820486e+07,0.802369


Лучший шаг: 0.001 / sqrt(1 + k)
Loss val: 15407482.52


12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [12]:
adam_const = search_step("adam", decay=False)
show_search(adam_const)


,lambda,Loss_train,Loss_val,R2_train
0,0.00001,4.534677e+07,4.155556e+07,0.254930
1,0.00010,1.987753e+07,2.042531e+07,0.673402
2,0.00100,1.206387e+07,1.789329e+07,0.801785
3,0.01000,1.202844e+07,1.869576e+07,0.802367
4,0.10000,1.203081e+07,1.868396e+07,0.802328
5,1.00000,1.811794e+07,2.254366e+07,0.702313


Лучший шаг: 0.001
Loss val: 17893285.35


13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [13]:
adam_decay = search_step("adam", decay=True)
show_search(adam_decay)


,lambda,Loss_train,Loss_val,R2_train
0,0.00001,5.984036e+07,5.443656e+07,0.016793
1,0.00010,5.181117e+07,4.724578e+07,0.148717
2,0.00100,2.440765e+07,2.384526e+07,0.598970
3,0.01000,1.212948e+07,1.769877e+07,0.800707
4,0.10000,1.202829e+07,1.871890e+07,0.802369
5,1.00000,1.202835e+07,1.863932e+07,0.802368


Лучший шаг: 0.01 / sqrt(1 + k)
Loss val: 17698768.51


14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [14]:
results = [
    ("VGD, постоянный", vgd_const),
    ("VGD, TimeDecay", vgd_decay),
    ("SGD, постоянный", sgd_const),
    ("SGD, TimeDecay", sgd_decay),
    ("SAG, постоянный", sag_const),
    ("SAG, TimeDecay", sag_decay),
    ("Momentum, постоянный", momentum_const),
    ("Momentum, TimeDecay", momentum_decay),
    ("Adam, постоянный", adam_const),
    ("Adam, TimeDecay", adam_decay),
]

summary = pd.DataFrame([{
    "Метод": name,
    "Лучший шаг": (f'{r["lambda"]} / sqrt(1+k)' if r["decay"] else str(r["lambda"])),
    "Loss_train": r["loss_train"],
    "Loss_test": r["loss_test"],
    "R2_train": r["r2_train"],
    "R2_test": r["r2_test"],
    "Итерации": r["iterations"],
} for name, r in results])
summary


,Метод,Лучший шаг,Loss_train,Loss_test,R2_train,R2_test,Итерации
0,"VGD, постоянный",0.001,1.336897e+07,2.996705e+07,0.780341,0.570288,1500
1,"VGD, TimeDecay",0.01 / sqrt(1+k),1.438496e+07,3.299997e+07,0.763648,0.526797,1500
2,"SGD, постоянный",0.001,1.345308e+07,3.039423e+07,0.778959,0.564162,1500
3,"SGD, TimeDecay",0.01 / sqrt(1+k),1.449348e+07,3.327258e+07,0.761865,0.522888,1500
4,"SAG, постоянный",0.0001,1.358157e+07,3.091278e+07,0.776848,0.556727,12000
5,"SAG, TimeDecay",0.01 / sqrt(1+k),1.317644e+07,2.911579e+07,0.783504,0.582494,12000
6,"Momentum, постоянный",0.0001,1.336842e+07,2.995433e+07,0.780350,0.570470,1500
7,"Momentum, TimeDecay",0.001 / sqrt(1+k),1.436261e+07,3.301635e+07,0.764015,0.526562,1500
8,"Adam, постоянный",0.001,1.206387e+07,3.164607e+07,0.801785,0.546212,1500
9,"Adam, TimeDecay",0.01 / sqrt(1+k),1.212948e+07,3.195228e+07,0.800707,0.541821,1500


15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

1. По качеству на тесте лучше всего получился SAG с TimeDecayLR, λ = 0.01: у него самый маленький Loss_test и самый высокий R^2_test. При этом он требует больше итераций.

2. R^2_train показывает, насколько хорошо модель объясняет данные, на которых училась. R^2_test показывает то же самое уже на новых данных. Чем значение ближе к 1, тем лучше.

3. Оба значения нужны, чтобы видеть не только качество обучения, но и то, как модель работает на новых данных. Если R^2_train высокий, а R^2_test заметно хуже, модель могла переобучиться.
